# Задание 8. Создание сентимент-анализатора

Наивный байесовский классификатор на отзывах о фильмах (корпус `movie_reviews` из NLTK). Признаки: наличие слова в отзыве. Классы: `Positive` и `Negative`.

In [8]:
import nltk
from nltk.corpus import movie_reviews
from nltk.classify import NaiveBayesClassifier
from nltk.classify.util import accuracy as nltk_accuracy

nltk.download("movie_reviews", quiet=True)

[nltk_data] Error loading movie_reviews: Security Violation
[nltk_data]     [pathsec.urlopen]: refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be SSRF-safe, opt in via
[nltk_data]     NLTK_ALLOW_PROXIED_URLOPEN=1 or
[nltk_data]     nltk.pathsec.ALLOW_PROXIED_FETCH=True.


False

In [9]:
# Загрузка помеченных отзывов
fileids_pos = movie_reviews.fileids("pos")
fileids_neg = movie_reviews.fileids("neg")

print("Положительных отзывов:", len(fileids_pos), "отрицательных:", len(fileids_neg))

Положительных отзывов: 1000 отрицательных: 1000


In [10]:
def extract_features(words):
    return dict([(word, True) for word in words])

features_pos = [(extract_features(movie_reviews.words(fileids=[f])), "Positive") for f in fileids_pos]
features_neg = [(extract_features(movie_reviews.words(fileids=[f])), "Negative") for f in fileids_neg]

In [11]:
# Разбиение 80/20 отдельно для положительных и отрицательных отзывов
num_pos = int(0.8 * len(features_pos))
num_neg = int(0.8 * len(features_neg))

features_train = features_pos[:num_pos] + features_neg[:num_neg]
features_test = features_pos[num_pos:] + features_neg[num_neg:]

print("Обучающая выборка:", len(features_train), "тестовая выборка:", len(features_test))

Обучающая выборка: 1600 тестовая выборка: 400


In [12]:
classifier = NaiveBayesClassifier.train(features_train)

accuracy = nltk_accuracy(classifier, features_test)
print("Accuracy на тестовой выборке:", round(accuracy, 4))

Accuracy на тестовой выборке: 0.735


In [13]:
# 10 наиболее информативных слов для Positive и 10 для Negative
# Берем признаки по величине отношения, затем направление определяем по вероятности класса для одного слова
cands = classifier.most_informative_features(500)
pos_words, neg_words = [], []
for word, _ratio in cands:
    p_pos = classifier.prob_classify({word: True}).prob("Positive")
    (pos_words if p_pos > 0.5 else neg_words).append((word, p_pos))

print("Positive (топ-10):")
for word, p in pos_words[:10]:
    print(f"   {word:<15}P(Positive)={p:.3f}")
print("Negative (топ-10):")
for word, p in neg_words[:10]:
    print(f"   {word:<15}P(Positive)={p:.3f}")

Positive (топ-10):
   outstanding    P(Positive)=0.933
   vulnerable     P(Positive)=0.929
   astounding     P(Positive)=0.921
   avoids         P(Positive)=0.921
   fascination    P(Positive)=0.917
   affecting      P(Positive)=0.912
   animators      P(Positive)=0.912
   anna           P(Positive)=0.912
   darker         P(Positive)=0.912
   symbol         P(Positive)=0.912
Negative (топ-10):
   insulting      P(Positive)=0.068
   ludicrous      P(Positive)=0.074
   uninvolving    P(Positive)=0.075
   seagal         P(Positive)=0.088
   idiotic        P(Positive)=0.093
   bothered       P(Positive)=0.100
   illogical      P(Positive)=0.100
   fairness       P(Positive)=0.107
   hudson         P(Positive)=0.107
   chuckle        P(Positive)=0.114


In [14]:
# Ручная выборка отзывов (по одному предложению), не входит в корпус
samples = [
    "A wonderful movie with brilliant acting and a moving story.",
    "The plot was boring and the characters were dull.",
    "I loved every minute of this film, it was fantastic.",
    "Terrible script, awful direction and a waste of time.",
    "The film is an enjoyable, well made comedy that I recommend.",
]

# Вероятности классов для каждого отзыва, выбор класса с максимальной вероятностью
for sentence in samples:
    prob = classifier.prob_classify(extract_features(sentence.split()))
    label = prob.max()
    print(f"{label:<9} P(Positive)={prob.prob('Positive'):.3f}  P(Negative)={prob.prob('Negative'):.3f}")
    print("          ", sentence)

Positive  P(Positive)=0.904  P(Negative)=0.096
           A wonderful movie with brilliant acting and a moving story.
Negative  P(Positive)=0.156  P(Negative)=0.844
           The plot was boring and the characters were dull.
Negative  P(Positive)=0.431  P(Negative)=0.569
           I loved every minute of this film, it was fantastic.
Negative  P(Positive)=0.047  P(Negative)=0.953
           Terrible script, awful direction and a waste of time.
Negative  P(Positive)=0.496  P(Negative)=0.504
           The film is an enjoyable, well made comedy that I recommend.
